# Generate the data that I need

In [1]:
import warnings
warnings.filterwarnings('ignore',category=RuntimeWarning)
import xarray as xr
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import glob,os,sys
from tqdm.auto import tqdm
import proplot as plot
import json,pickle
import dask.array as da
import gc
from sklearn.decomposition import PCA
from tools import derive_var,read_and_proc,preproc_noensemble
from tools.mlr import mlr
from tools.preprocess import do_eof,preproc_maria,preproc_haiyan
from tqdm.auto import tqdm
import random
import data_process
%matplotlib inline
plot.rc.update({'figure.facecolor':'w','axes.labelweight':'ultralight',
                'tick.labelweight':'ultralight','gridminor.linestyle':'--','title.weight':'normal','linewidth':0.5})
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:32"

/tmp/ipykernel_866432/3854849951.py:22: ProplotWarning: rc setting 'linewidth' was renamed to 'meta.width' in version 0.8.
  plot.rc.update({'figure.facecolor':'w','axes.labelweight':'ultralight',


## Design 5 folds

In [5]:
# --- Setup ---
n_members = 20
independent_test = {10,  17}   # excluded members
val_size = 4
random.seed(42)               # for reproducibility; remove or change for new splits

# --- Eligible members for cross-validation ---
members = [m for m in range(0, n_members) if m not in independent_test]
random.shuffle(members)

n_folds = int(np.ceil(len(members) / val_size))

# --- Generate folds ---
folds = []
for i in range(n_folds):
    start = i * val_size
    end = start + val_size
    val_members = members[start:end]

    # If last fold has fewer than val_size, wrap around
    if len(val_members) < val_size:
        val_members += members[:val_size - len(val_members)]

    folds.append(val_members)

# --- Print results ---
for i, f in enumerate(folds, 1):
    print(f"Fold {i}: validation members = {sorted(f)}")

Fold 1: validation members = [4, 9, 13, 14]
Fold 2: validation members = [5, 6, 7, 18]
Fold 3: validation members = [1, 11, 12, 16]
Fold 4: validation members = [2, 8, 15, 19]
Fold 5: validation members = [0, 3, 13, 14]


In [8]:
(list(independent_test))

[17, 10]

## Function

In [16]:
class proc_inputoutput:
    def __init__(self,validindices=None,testindices=None,pcastorepath=None):
        self.validindices=validindices
        self.testindices=testindices
        self.pcastorepath=pcastorepath #'./store/pca/'
        
    def forward_diff(self,arrayin=None,delta=None,axis=None,LT=1):
        if len(arrayin.shape)>1:
            result = []
            if axis==0:
                for i in range(0,arrayin.shape[axis]-LT):
                    temp = (arrayin[i+LT,:]-arrayin[i,:])/(LT*delta)
                    result.append(temp)
                return np.asarray(result)
        else:
            result = []
            for i in range(0,arrayin.shape[axis]-LT):
                temp = (arrayin[i+LT]-arrayin[i])/(LT*delta)
                result.append(temp)
            return np.asarray(result)
        
    def _get_time_diff_ts(self,array=None,timedelta=60*60,LT=None):
        store = []
        for exp in array: 
            a = self.forward_diff(exp,timedelta,0,LT)
            if a.shape[0]>0:
                azero = np.zeros((LT))
                store.append(np.concatenate((a,azero),axis=0))
            else:
                store.append(np.zeros((exp.shape[0])))
        return store
    
    def myPCA_projection_sen(self,pca_dict=None,varname=None,toproj_flatvar=None,orig_flatvar=None):
        projvar_transformed = np.dot(toproj_flatvar-np.nanmean(orig_flatvar,axis=0),pca_dict[varname].components_.T)
        return projvar_transformed

    def create_timeseries(self,var=None,varname=None,splitnum=None):
        Xtrain,Xvalid,Xtest = preproc.train_valid_test(var,self.validindices,self.testindices)#[int(self.indices[splitnum][0]),int(self.indices[splitnum][1])],[int(self.indices[splitnum][2]),int(self.indices[splitnum][3])],'Yes')
        pca = read_and_proc.depickle(glob.glob(self.pcastorepath+str(varname)+'/'+str(splitnum)+'/*')[0])
        train = pca[varname].transform(Xtrain)
        valid = self.myPCA_projection_sen(pca,varname,Xvalid,Xtrain)
        test = self.myPCA_projection_sen(pca,varname,Xtest,Xtrain)
        timeseries = {'train':train,'valid':valid,'test':test}
        return timeseries
    
    def normalize_timeseries(self,timeseries=None,category='train'):
        #assert timeseries['u'].shape[-1]==26,"var shape error"
        output = np.zeros_like(timeseries[category])
        for le in range(timeseries[category].shape[1]):
            trainmean,trainstd = np.nanmean(timeseries['train'][:,le]), np.nanstd(timeseries['train'][:,le])
            output[:,le] = (timeseries[category][:,le]-trainmean)/trainstd
        return output
    
    def normalize_timeseries_sensitivity(self,timeseries=None,category='train'):
        #assert timeseries['u'].shape[-1]==26,"var shape error"
        output = np.zeros_like(timeseries[category])
        for le in range(timeseries[category].shape[1]):
            trainmean,trainstd = np.nanmean(timeseries['train'][:,le]), np.nanstd(timeseries['train'][:,le])
            output[:,le] = (timeseries[category][:,le]-trainmean)/trainstd
        return output
    
    def train_valid_test(self,listt=None,splitnum=None):
        #valid, test = [listt[i] for i in [int(self.indices[splitnum][0]),int(self.indices[splitnum][1])]], [listt[i] for i in [int(self.indices[splitnum][2]),int(self.indices[splitnum][3])]]
        valid, test = [listt[i] for i in self.validindices], [listt[i] for i in self.testindices]
        #popindex = [int(self.indices[splitnum][0]),int(self.indices[splitnum][1])]+[int(self.indices[splitnum][2]),int(self.indices[splitnum][3])]
        popindex = self.validindices+self.testindices
        train = [listt[i] for i in range(len(listt)) if i not in popindex]
        return train, valid, test
    
    def _back_to_exp(self,timeseries=None,divider=None):
        if len(timeseries.shape)==2:
            printout = [timeseries[0:divider[0],:]]
            for i in range(1,len(divider)-2):
                printout.append(timeseries[divider[i-1]:divider[i],:])
            printout.append(timeseries[divider[-2]:,:])
        elif len(timeseries.shape)==1:
            printout = [timeseries[0:divider[0]]]
            for i in range(1,len(divider)-2):
                printout.append(timeseries[divider[i-1]:divider[i]])
            printout.append(timeseries[divider[-2]:])            
        return printout
    
    def back_to_exp(self,inputlong=None,divider=None,senvarname=None):
        ts_dict = {}
        if senvarname is None:
            for indx,obj in tqdm(enumerate(self.varname)):
                ts_dict[obj] = self._back_to_exp(inputlong[obj],divider)
        else:
            for indx,obj in tqdm(enumerate(senvarname)):
                ts_dict[obj] = self._back_to_exp(inputlong[obj],divider)            
        return ts_dict
    
    def create_X(self,vardicts=None,nummem=None,varnames=None,splitnum=None):
        trains,valids,tests = {},{},{}
        for ind,obj in enumerate(varnames):
            timeseries = self.create_timeseries(vardicts[obj],obj,splitnum)
            trains[obj] = self.normalize_timeseries(timeseries,'train')[:,:nummem[ind]]
            valids[obj] = self.normalize_timeseries(timeseries,'valid')[:,:nummem[ind]]
            tests[obj] = self.normalize_timeseries(timeseries,'test')[:,:nummem[ind]]
        return trains,valids,tests 

import torch.nn as nn

def weights_init(m):
    if isinstance(m, nn.Linear) or isinstance(m, nn.Conv3d):
        nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    if hasattr(m, "weight") and m.weight is not None and "logvar" in m.__class__.__name__.lower():
        with torch.no_grad():
            m.weight *= 0.1  # dampen logvar layers

def get_max_intensity(u=None,v=None,shape=[10,360,208]):
    TEMPts = np.max(np.mean(np.sqrt(v.reshape(v.shape[0],shape[0],shape[1],shape[2])[:,0,...]**2+u.reshape(u.shape[0],shape[0],shape[1],shape[2])[:,0,...]**2),axis=1),axis=1)
    return TEMPts

## Output

In [6]:
def get_max_intensity(u=None,v=None,shape=[10,360,208]):
    TEMPts = np.max(np.mean(np.sqrt(v.reshape(v.shape[0],shape[0],shape[1],shape[2])[:,0,...]**2+u.reshape(u.shape[0],shape[0],shape[1],shape[2])[:,0,...]**2),axis=1),axis=1)
    return TEMPts

In [8]:
name='HAIYAN'
path = '/work/FAC/FGSE/IDYST/tbeucler/default/freddy0218/'
suffix = '_smooth_preproc_dict1b_g'
haiyan_u = [read_and_proc.depickle(path+'TCGphy/2020_TC_CRF/dev/freddy0218/testML/output/haiyan/processed/uvwheat/'+'mem'+str(lime)+suffix)['u'] for lime in tqdm(range(1,21))]
haiyan_v = [read_and_proc.depickle(path+'TCGphy/2020_TC_CRF/dev/freddy0218/testML/output/haiyan/processed/uvwheat/'+'mem'+str(lime)+suffix)['v'] for lime in tqdm(range(1,21))]
    
storeWSPD = []
for i in range(len(haiyan_u)):
    storeWSPD.append(get_max_intensity(haiyan_u[i],haiyan_v[i]))

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

In [10]:
read_and_proc.save_to_pickle('./storeWSPDn.pkl',storeWSPD)

## input

In [10]:
name='HAIYAN'

In [11]:
import importlib
importlib.reload(data_process)

<module 'data_process' from '/work/FAC/FGSE/IDYST/tbeucler/default/freddy0218/2024_TCG_VED_WRFsen/haiyan/data_process.py'>

In [12]:
if name=='HAIYAN':
    path = '/work/FAC/FGSE/IDYST/tbeucler/default/freddy0218/'
    suffix = '_smooth_preproc_dict1b_g'
    suffixRAD = '_smooth_preproc_dict1b_g_radcomp'
    haiyan_lw = [read_and_proc.depickle(path+'TCGphy/2020_TC_CRF/dev/freddy0218/testML/output/haiyan/processed/uvwheat/radcomp/'+'mem'+str(lime)+suffixRAD)['LW'] for lime in tqdm(range(1,21))]
    pfull_array = [data_process.get_pfull(haiyan_lw[lime]) for lime in tqdm(range(20))]
    
    adjusted_lw = [data_process.get_adjusted_rads(haiyan_lw[i],pfull_array[i])[0].reshape(haiyan_lw[i].shape[0],-1) for i in range(20)]
    del haiyan_lw
    gc.collect()

    for exp in tqdm(range(len(folds))):
        Xtrain,Xvalid,Xtest = data_process.train_valid_test(adjusted_lw,folds[exp],sorted(list(independent_test)))
        Xtrain_cart = Xtrain.reshape(-1,10,360,208)
        Xvalid_cart = Xvalid.reshape(-1,10,360,208)
        Xtest_cart = Xtest.reshape(-1,10,360,208)
        read_and_proc.save_to_pickle(f'./X_cart_newadjustedLW_{exp}.pkl',{'train':Xtrain_cart,'valid':Xvalid_cart,'test':Xtest_cart},'PICKLE')

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
import read_stuff as read
# --- Verification ---
# Recompute total volume-integrated heating
dtheta = np.deg2rad(1.0)
n_p, n_r, n_theta = TEMPTEMP.shape
r = (np.arange(n_r) + 0.5) * 3000.0
dp = pfull_array[0][50][:-1, :, :] - pfull_array[0][50][1:, :, :]
if dp.shape[0] != TEMPTEMP.shape[0]:
    dp = np.pad(dp, ((0,1),(0,0),(0,0)), mode='edge')

R = r[:, None]
weight_3d = dp * (R[None, :, :] * 3000.0 * dtheta)

total_before = np.sum(TEMPTEMP * weight_3d)
total_after = np.sum(ADJTEMPTEMP * weight_3d)

# Relative error
rel_error = abs(total_after / total_before)
print(f"Relative error after correction: {rel_error:.3e}")
